# Plan: Creating a model predicting a sequence of digits from an image

## Overall plan

Image → CNN → Sequence Features → BiLSTM → CTC → Digit String

# Loading the required dependencies

In [19]:
import h5py
import numpy as np
import cv2
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Loading the dataset

We are using the SVHN dataset which consists of sequence of digits collected by google street view images

Link to the dataset: https://www.kaggle.com/datasets/stanfordu/street-view-house-numbers

## Helper function to read strings from HDF5 data

In [20]:
def h5py_string(h5file, ref):
    data = h5file[ref][()]
    data = data.flatten()
    return ''.join(chr(int(c)) for c in data)

In [21]:
def to_scalar(x):
    return int(np.array(x).squeeze())

## Extract bounding box data

In [22]:
def get_bbox_data(h5file, bbox_ref):
    bbox = h5file[bbox_ref]

    def extract(name):
        values = []
        refs = h5file[bbox_ref][name]
    
        for ref in refs:
            ref = ref[0]  # important: unwrap array layer
    
            if isinstance(ref, h5py.Reference):
                val = h5file[ref][()]
            else:
                # sometimes SVHN stores direct values
                val = ref
    
            val = np.array(val).squeeze()
            values.append(int(val))
    
        return values

    labels = extract('label')
    lefts = extract('left')
    tops = extract('top')
    widths = extract('width')
    heights = extract('height')

    labels = [0 if l == 10 else l for l in labels]

    return list(zip(labels, lefts, tops, widths, heights))

## Load the entire dataset metadata

In [23]:
def load_svhn_h5(mat_path):
    h5file = h5py.File(mat_path, 'r')
    digitStruct = h5file['digitStruct']

    names = digitStruct['name']
    bboxes = digitStruct['bbox']

    samples = []

    for i in range(len(names)):
        name_ref = names[i][0]
        filename = h5py_string(h5file, name_ref)

        bbox_ref = bboxes[i][0]
        digits = get_bbox_data(h5file, bbox_ref)

        samples.append((filename, digits))

    return samples

# Image processing

## Cropping the full number

In [24]:
def crop_number(img, digits):
    h, w = img.shape[:2]

    xs = []
    ys = []

    for d in digits:
        left = int(d[1])
        top = int(d[2])
        width = int(d[3])
        height = int(d[4])

        xs.append(left)
        xs.append(left + width)
        ys.append(top)
        ys.append(top + height)

    x1 = max(0, min(xs))
    y1 = max(0, min(ys))
    x2 = min(w, max(xs))
    y2 = min(h, max(ys))

    # ✅ safety check
    if x2 <= x1 or y2 <= y1:
        return None

    cropped = img[y1:y2, x1:x2]

    if cropped.size == 0:
        return None

    return cropped

## Resize + Normalize

In [25]:
def preprocess(img):
    if img is None or img.size == 0:
        raise ValueError("Empty image in preprocess")

    # ✅ handle grayscale safely
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    h = 32
    scale = h / img.shape[0]
    img = cv2.resize(img, None, fx=scale, fy=scale)

    return img, img.shape[1]

## Padding

In [26]:
def pad_image(img, max_width=128):
    padded = np.zeros((32, max_width))
    padded[:, :img.shape[1]] = img
    return padded

## Pytorch Dataset

In [27]:
class SVHNDataset(Dataset):
    def __init__(self, img_dir, mat_path, max_width=128):
        self.img_dir = img_dir
        self.samples = load_svhn_h5(mat_path)
        self.max_width = max_width

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, digits = self.samples[idx]

        img_path = os.path.join(self.img_dir, filename)
        img = cv2.imread(img_path)

        if img is None:
            raise ValueError(f"Image not found: {img_path}")

        # crop full number
        img = crop_number(img, digits)

        if img is None or img.size == 0:
            raise ValueError(f"Crop failed for: {img_path}")

        # preprocess
        img, width = preprocess(img)

        if width > self.max_width:
            width = self.max_width
            img = img[:, :self.max_width]

        img = pad_image(img, self.max_width)

        label = [d[0] for d in digits]

        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        label = [int(d[0]) if isinstance(d, (list, tuple, np.ndarray)) else int(d) for d in digits]

        return img, label, width

# Collate function (CTC ready)

In [28]:
def collate_fn(batch):
    images, labels, widths = zip(*batch)

    images = torch.stack(images)

    # ensure all labels are tensors
    labels = [l.clone().detach() if isinstance(l, torch.Tensor) else torch.tensor(l) for l in labels]

    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)

    labels = torch.cat(labels, dim=0)

    input_lengths = torch.tensor([w // 4 for w in widths], dtype=torch.long)

    return images, labels, input_lengths, label_lengths

# Data loader

In [29]:
dataset = SVHNDataset(
    img_dir="../data/train/train",
    mat_path="../data/train_digitStruct.mat"
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

# Testing the data

In [30]:
for images, labels, input_lengths, label_lengths in loader:
    print(images.shape)
    print(labels.shape)
    print(input_lengths)
    print(label_lengths)
    break

torch.Size([32, 1, 32, 128])
torch.Size([72])
tensor([ 7,  7,  3,  8, 10,  2,  6, 18, 17, 11,  8,  2,  7,  8,  7,  8,  6,  7,
         8,  5, 17, 10,  4,  4,  9,  9, 12, 10,  8, 10,  9, 10])
tensor([2, 2, 1, 2, 3, 1, 2, 2, 4, 3, 2, 1, 3, 2, 2, 2, 2, 3, 2, 2, 4, 2, 1, 2,
        2, 2, 2, 3, 3, 3, 2, 3])


# Building the CRNN architecture 

In [31]:
class CRNN(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(128,256,3,padding=1), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(256,512,3,padding=1), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(512,512,2)
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)          # (B, C, 1, W)
        x = x.squeeze(2)         # (B, C, W)
        x = x.permute(0, 2, 1)   # (B, W, C)

        x, _ = self.rnn(x)
        x = self.fc(x)           # (B, W, num_classes)

        return x

# Setup

In [32]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN().to(device)

criterion = nn.CTCLoss(blank=10, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)

# One training step

In [33]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels, input_lengths, label_lengths in loader:
        images = images.to(device)
        labels = labels.to(device)

        preds = model(images)              # (B, W, C)
        preds = preds.permute(1, 0, 2)     # (W, B, C) for CTC

        loss = criterion(
            preds.log_softmax(2),         # IMPORTANT
            labels,
            input_lengths,
            label_lengths
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# Decoding

In [34]:
def decode(preds):
    preds = preds.argmax(dim=2)

    results = []
    for seq in preds:
        prev = -1
        decoded = []

        for p in seq:
            if p != prev and p != 10:  # remove repeats + blank
                decoded.append(str(p.item()))
            prev = p

        results.append("".join(decoded))

    return results

# Evaluation

In [35]:
def evaluate(model, loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, _, label_lengths in loader:
            images = images.to(device)

            preds = model(images)              # (B, W, C)
            decoded = decode(preds)

            idx = 0
            for i, length in enumerate(label_lengths):
                true = labels[idx:idx+length].tolist()
                true = "".join(map(str, true))

                if decoded[i] == true:
                    correct += 1

                total += 1
                idx += length

    print("Accuracy:", correct / total)

# Run training

In [36]:
EPOCHS = 10

for epoch in range(EPOCHS):
    loss = train_one_epoch(model, loader)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

    evaluate(model, loader)

RuntimeError: Expected input_lengths to have value at most 31, but got value 32 (while checking arguments for ctc_loss_gpu)